# MIMIC-IV Data for Temporal Domain Generalization

This notebook uses the modified `mimic_iv_Version2` module to load and explore MIMIC-IV data for mortality prediction across temporal splits (2008–2024).

In [7]:
import os
import sys
import numpy as np
import pandas as pd

# Add workspace directory to path so mimic_iv_Version2 and domain_validation can be imported
_WORKSPACE = os.getcwd()
if _WORKSPACE not in sys.path:
    sys.path.insert(0, _WORKSPACE)
if 'mimic_iv_Version2' in sys.modules:
    del sys.modules['mimic_iv_Version2']
if 'domain_validation' in sys.modules:
    del sys.modules['domain_validation']
from mimic_iv_Version2 import (
    MIMICIVDataset,
    preprocess_mimic_iv,
    get_stay_dict,
    TIME_PERIODS,
    TEST_SPLIT,
)
from domain_validation import (
    run_domain_validation,
    compute_avg_probs_per_source_domain,
    plot_domain_probability_lines,
)

In [9]:
# Data directory: Data folder in workspace
# Expects: admissions.csv(.gz), patients.csv(.gz), diagnoses_icd.csv(.gz)
DATA_DIR = os.path.join(_WORKSPACE, 'Data')
print(f"Data directory: {DATA_DIR}")

Data directory: /Users/oliverli/Documents/26Spring/BM_Data_Design/MIMIC/Data


In [10]:
# Load dataset (preprocesses automatically if mimic_iv_wildtime.pkl does not exist)
# Use force_reprocess=True to rebuild from raw CSVs
dataset_train = MIMICIVDataset(DATA_DIR, split='train')
dataset_test = MIMICIVDataset(DATA_DIR, split='test')

Loaded MIMIC-IV data with 5 time periods: [2008, 2011, 2014, 2017, 2020]
Period 2008: 79529 samples (1407 positive, 1.8% mortality)
Period 2011: 89787 samples (1516 positive, 1.7% mortality)
Period 2014: 89789 samples (1714 positive, 1.9% mortality)
Period 2017: 87100 samples (1877 positive, 2.2% mortality)
Period 2020: 67648 samples (1875 positive, 2.8% mortality)
Loaded MIMIC-IV data with 5 time periods: [2008, 2011, 2014, 2017, 2020]
Period 2008: 19977 samples (324 positive, 1.6% mortality)
Period 2011: 22647 samples (387 positive, 1.7% mortality)
Period 2014: 22305 samples (413 positive, 1.9% mortality)
Period 2017: 21679 samples (464 positive, 2.1% mortality)
Period 2020: 16849 samples (447 positive, 2.7% mortality)


In [11]:
# Period statistics (for domain validation / TDG analysis)
stats_train = pd.DataFrame(dataset_train.get_period_stats())
stats_test = pd.DataFrame(dataset_test.get_period_stats())
print("Train split:")
display(stats_train)
print("\nTest split:")
display(stats_test)

Train split:


,period,n_samples,n_positive,mortality_rate
0,2008,79529,1407,1.769166
1,2011,89787,1516,1.688440
2,2014,89789,1714,1.908920
3,2017,87100,1877,2.154994
4,2020,67648,1875,2.771701



Test split:


,period,n_samples,n_positive,mortality_rate
0,2008,19977,324,1.621865
1,2011,22647,387,1.708836
2,2014,22305,413,1.851603
3,2017,21679,464,2.140320
4,2020,16849,447,2.652976


## Domain Validation (Domain Discriminator)

We implement a **Domain Discriminator** $D: X \to T$ that predicts time period $T$ from patient features $X$ (diagnoses, age, gender, ethnicity). This empirically validates that temporal distribution shifts exist in the data.

- **Proof of shift**: If the discriminator can accurately identify the domain (e.g., $P(T=2008|X)$ is highest for 2008 data), it confirms that features carry a "temporal signature."
- **Quantifying drift**: Higher discriminator accuracy → more severe distribution shift → more critical the Kalman Filter's trajectory tracking.

In [12]:
# Run domain validation: train multi-class classifier D: X -> T to predict time period from features
# Uses domain_validation.py (prepare_domain_validation_data, DomainDiscriminator, train_domain_discriminator)
results, feature_info = run_domain_validation(
    DATA_DIR,
    max_samples_per_period=50000,  # subsample for speed; set None for full data
    epochs=20,
    batch_size=256,
)

Loading data for domain validation...
Train: 200000, Test: 50000, Features: 2036
Domains (periods): [2008, 2011, 2014, 2017, 2020]

Training domain discriminator...
  Epoch 5/20, loss=0.8271
  Epoch 10/20, loss=0.7005
  Epoch 15/20, loss=0.6183
  Epoch 20/20, loss=0.5692

DOMAIN VALIDATION RESULTS
 discriminator accuracy: 54.63%
 random baseline (1/5): 20.00%

CONCLUSION: Discriminator predicts time period above chance.
  -> Temporal distribution shift is CONFIRMED.
  -> Features carry a 'temporal signature'.
  -> Kalman Filter trajectory tracking is justified.


In [13]:
# Per-period accuracy (how well each domain is predicted)
y_true = results['true_labels']
y_pred = results['predictions']
idx_to_period = feature_info['idx_to_period']

print("Per-period accuracy (how well each domain is predicted):")
for idx in range(len(TIME_PERIODS)):
    period = idx_to_period[idx]
    mask = y_true == idx
    if mask.sum() > 0:
        acc = (y_pred[mask] == idx).mean()
        n = mask.sum()
        print(f"  Period {period} ({period}-{period+2}): {acc:.1%} correct (n={n})")

Per-period accuracy (how well each domain is predicted):
  Period 2008 (2008-2010): 64.1% correct (n=10036)
  Period 2011 (2011-2013): 46.0% correct (n=9991)
  Period 2014 (2014-2016): 34.1% correct (n=10109)
  Period 2017 (2017-2019): 64.9% correct (n=9827)
  Period 2020 (2020-2022): 64.5% correct (n=10037)


### Domain probability plot

For each source domain, average the predicted probabilities across all test samples from that domain. X axis: time domain. Y axis: probability. Each line = features from a different domain.

In [ ]:
# Compute average probabilities per source domain (test set, option b: average over all samples)
avg_probs = compute_avg_probs_per_source_domain(results, feature_info)

# Plot: X axis = time domain, Y axis = probability, each line = features from a different domain
import matplotlib.pyplot as plt
plot_domain_probability_lines(avg_probs, feature_info)
plt.show()

### Domain probability plot

For each source domain, average the predicted probabilities across all test samples from that domain. X axis: time domain. Y axis: probability. Each line = features from a different domain.